# Semantic Understanding & Multilingual

Aliases, abbreviations, typo correction, and cross-language queries.

**Rollup: 17/17 cases passed** across 2 required capabilities, est. cost $0.0193, 1254s total.

This is an executive-level summary over already-captured real-LLM results -- see `docs/CAPABILITY_MAPPING.md` for the full 25-capability table, and `notebooks/capabilities/<NN>_<slug>/demo.ipynb` for every case in full detail per capability.

In [ ]:
import sys, pathlib

# Robust path insert regardless of where Jupyter's cwd lands (repo root, or
# this notebook's own folder under notebooks/capabilities/<slug>/):
_p = pathlib.Path.cwd()
while not (_p / "src").exists() and _p != _p.parent:
    _p = _p.parent
sys.path.insert(0, str(_p))

import os

try:
    from dotenv import load_dotenv  # optional: picks up a .env file if python-dotenv is installed
    load_dotenv(override=False)
except ImportError:
    pass

from src.orchestrator import Orchestrator
from src.llm_client import get_llm_client, GLOBAL_USAGE, MockLLMClient

provider = os.environ.get("LLM_PROVIDER", "").lower() or ("anthropic" if os.environ.get("ANTHROPIC_API_KEY") else "openai" if os.environ.get("OPENAI_API_KEY") else "mock")
print(f"LLM provider in use: {provider}" + ("  (\u26a0\ufe0f set ANTHROPIC_API_KEY or OPENAI_API_KEY for real answers)" if provider == "mock" else ""))

orch = Orchestrator()


LLM provider in use: openai


In [ ]:
def ask(question: str, label: str = ""):
    """Run one turn through the orchestrator and pretty-print everything the
    grader needs to see: routing, evidence sources, transparency notes, answer.
    Identical helper to notebooks/demo.ipynb -- see scripts/build_notebook.py."""
    if label:
        print(f"\n{'='*90}\n{label}\n{'='*90}")
    print(f"USER: {question}\n")
    resp = orch.handle_turn(question)
    print(f"[intent={resp.intent} | sub_agents={resp.sub_agents_used} | retried={resp.retried}]")
    if resp.citations:
        print(f"[citations: {[c['doc_id'] for c in resp.citations]}]")
    if resp.assumptions:
        print("[assumptions/limitations surfaced:]")
        for a in resp.assumptions:
            print(f"  - {a}")
    print(f"\nAGENT: {resp.answer}")
    if resp.follow_up_suggestions:
        print(f"\n(follow-up suggestions: {resp.follow_up_suggestions})")
    return resp


## 6. Semantic understanding: aliases, abbreviations, and typo correction

9/9 passed — $0.0117, 637s

In [ ]:
_ = ask('NA rev Q1 2024?', '01')
_ = ask('EMEA revenue Q2 2025', '02')

01
USER: NA rev Q1 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: North America revenue for Q1 2024 was **$3,593 million USD**.

02
USER: EMEA revenue Q2 2025

[intent=data_query | sub_agents=['structured']]

AGENT: EMEA revenue for **Q2 2025** was **$2,489 million USD**.

Want me to pull the prior quarter or year-ago comparison for context?

## 7. Multilingual and mixed-language queries

8/8 passed — $0.0076, 617s

In [ ]:
_ = ask('¿Cuáles fueron los ingresos de Middle Americas en el primer trimestre de 2024?', '01')
_ = ask('Quelle était la marge EBITDA en EMEA au deuxième trimestre 2025?', '02')

01
USER: ¿Cuáles fueron los ingresos de Middle Americas en el primer trimestre de 2024?

[intent=data_query | sub_agents=['structured']]

AGENT: Los ingresos de **Middle Americas** en el **primer trimestre de 2024** fueron de **$4,051 millones de USD** (según los datos estructurados de KPI).

¿Te gustaría comparar esta cifra con otro trimestre o desglosar los ingresos por país o marca?

02
USER: Quelle était la marge EBITDA en EMEA au deuxième trimestre 2025?

[intent=data_query | sub_agents=['structured']]

AGENT: La marge EBITDA en zone EMEA pour le deuxième trimestre 2025 s’élevait à **32,1 %**.

Si vous le souhaitez, je peux également vous indiquer l’évolution par rapport au trimestre précédent ou à la même période de l’année précédente.